# 05 · Limpieza de producción real

**Entra:** seis libros anuales originales de `Datos/Producción real/`.
**Sale:** una única tabla de movimientos: `Datos_analiticos_proyecto/produccion_real_limpio.parquet`.

Una fila conserva un registro del Excel, con archivo, hoja y fila. No borramos movimientos por parecer duplicados; no imputamos cantidades ni tipos de corte. El semanal se construye después, en el notebook 06. La unidad es **tallos reportados**, no exportables sin validación adicional.

## 1. Imports y rutas

Las utilidades compartidas sólo normalizan texto/fechas; la limpieza se ve a continuación.

In [1]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Datos").is_dir() and (p / "Notebooks").is_dir())
from datetime import date
import unicodedata
from pathlib import PurePosixPath
from zipfile import ZipFile
from lxml import etree

def texto(valor):
    if pd.isna(valor):
        return ""
    s = unicodedata.normalize("NFKD", str(valor).replace("\xa0", " "))
    return re.sub(r"\s+", " ", "".join(c for c in s if not unicodedata.combining(c))).strip().upper()

def normalizar(serie):
    return (serie.astype("string").str.normalize("NFKD")
            .str.replace("[\u0300-\u036f]", "", regex=True)
            .str.replace(r"\s+", " ", regex=True).str.strip().str.upper()
            .replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "<NA>": pd.NA}))

def finca(serie):
    return normalizar(serie).replace({"GF": "GAITANA", "LA GAITANA": "GAITANA", "¡": "GAITANA",
                                     "AR": "ARABELLA", "LA CABANA": "CABANA", "FINCA LA CABANA": "CABANA"})

def fechas(serie):
    """Conservar calendario de Excel: fechas ya leídas o seriales 1900, no nanosegundos 1970."""
    salida = pd.to_datetime(serie, errors="coerce", format="mixed")
    numeros = pd.to_numeric(serie.astype("string"), errors="coerce")
    serial = numeros.between(20000, 80000)
    salida.loc[serial] = pd.to_datetime(numeros[serial], unit="D", origin="1899-12-30")
    return salida

def semana_operativa(anio, semana):
    """Lunes representativo. Toda etiqueta 53 se convierte en 01 del año siguiente."""
    y = pd.to_numeric(anio, errors="coerce").astype("Int64")
    w = pd.to_numeric(semana, errors="coerce").astype("Int64")
    y = y + w.eq(53).fillna(False).astype(int)
    w = w.mask(w.eq(53), 1)
    etiquetas = y.astype("string") + "-" + w.astype("string")
    conversion = {}
    for etiqueta in etiquetas.dropna().unique():
        try:
            yy, ww = map(int, etiqueta.split("-"))
            conversion[etiqueta] = pd.Timestamp(date.fromisocalendar(yy, ww, 1)) if 1 <= ww <= 52 else pd.NaT
        except (ValueError, OverflowError):
            conversion[etiqueta] = pd.NaT
    return pd.to_datetime(etiquetas.map(conversion))

SALIDA = ROOT / "Datos_analiticos_proyecto"
SALIDA.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 35)
print("Fuentes originales:", ROOT / "Datos")
print("Salida oficial:", SALIDA)

ENTRADA = ROOT / "Datos/Producción real"
ARCHIVO_SALIDA = SALIDA / "produccion_real_limpio.parquet"
HOJAS = {2021:["Sheet1"], 2022:["Sheet1"], 2023:["Sheet1","Hoja1"],
         2024:["Sheet1","sem 45 en adelante"], 2025:["01-44","45-52"], 2026:["BD"]}
archivos = sorted(ENTRADA.glob("*.xlsx"))
display(pd.DataFrame({"archivo":[p.name for p in archivos],"MB":[round(p.stat().st_size/1e6,1) for p in archivos]}))
assert len(archivos) >= 6


Fuentes originales: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos
Salida oficial: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto


,archivo,MB
0,Producción Acumulada 2021.xlsx,67.9
1,Producción Acumulada 2022.xlsx,74.5
2,Producción Acumulada 2023.xlsx,93.5
3,Producción Acumulada 2024.xlsx,112.4
4,Producción Acumulada 2025.xlsx,113.4
5,Producción Acumulada 2026.xlsx,85.2


## 2. Estructura original y selección de columnas

Las hojas resumen no son movimientos y no se suman al detalle. Se preservan las 15 columnas productivas usadas históricamente; quedan visibles tanto las etiquetas de año/semana originales como las calculadas desde la fecha.

In [2]:
COLUMNAS = {"FINCA":"finca","INVERNADERO":"invernadero","FECHA CORTE":"fecha_corte",
"ANO":"anio_fuente","SEMANA":"semana_fuente","TIPO CORTE":"tipo_corte",
"PRODUCTO":"producto","GRADO":"grado","COLOR":"color","VARIEDAD":"variedad",
"TALLOS":"tallos","LOTE":"lote","ESTADO INVENTARIO":"estado_inventario",
"HORA CORTE":"hora_corte","TIPO INVENTARIO":"tipo_inventario"}
primero = pd.read_excel(archivos[0],sheet_name="Sheet1",nrows=5,engine="calamine")
display(primero)
print("Columnas originales:",primero.columns.tolist())


,Finca,Sector,Invernadero,Fecha Corte,Hora Corte,Tipo Corte,Año,Semana,Lote,Producto,Grado,Color,Variedad,Tallos,Estado Inventario,Unnamed: 15,Unnamed: 16,Unnamed: 17
0,Arabella,Amalia Rodriguez,Bloque 01,2021-01-04,2:55PM,Corte 2,2021,1,9910,Carnation,NaN,burgundy,Zurigo,30,C,NaN,NaN,NaN
1,Arabella,Amalia Rodriguez,Bloque 01,2021-01-04,2:56PM,Corte 2,2021,1,15007,Carnation,NaN,purple,Extasis,8,A,NaN,NaN,NaN
2,Arabella,Amalia Rodriguez,Bloque 01,2021-01-04,2:56PM,Corte 2,2021,1,18156,Carnation,NaN,bicolor burgundy,Olympia,32,C,NaN,NaN,NaN
3,Arabella,Amalia Rodriguez,Bloque 01,2021-01-04,2:56PM,Corte 2,2021,1,36397,Carnation,NaN,yellow,Diletta,12,C,NaN,NaN,NaN
4,Arabella,Amalia Rodriguez,Bloque 01,2021-01-04,2:58PM,Corte 2,2021,1,16984,Carnation,NaN,gold,Caroline Gold,23,C,NaN,NaN,NaN


Columnas originales: ['Finca', 'Sector', 'Invernadero', 'Fecha Corte', 'Hora Corte', 'Tipo Corte', 'Año', 'Semana', 'Lote', 'Producto', 'Grado', 'Color', 'Variedad', 'Tallos', 'Estado Inventario', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17']


## 3. Leer y limpiar cada hoja

Sólo se retiran filas completamente vacías en las columnas productivas. No se usa `drop_duplicates` sobre movimientos. Las cantidades se contrastan antes/después de limpiar. Se conserva el alias documentado `¡ → GAITANA` para no perder sus 80 tallos.

In [3]:
partes, controles = [], []
for archivo in archivos:
    anio = int(re.search(r"20\d{2}",archivo.name).group())
    if anio not in HOJAS:
        raise ValueError(f"Definir explícitamente las hojas de detalle del año {anio}")
    libro = pd.ExcelFile(archivo,engine="calamine")
    for hoja in HOJAS[anio]:
        df = pd.read_excel(libro,sheet_name=hoja,usecols=lambda c:texto(c) in COLUMNAS)
        df.columns = [COLUMNAS[texto(c)] for c in df.columns]
        assert {'finca','fecha_corte','producto','color','variedad','tallos'} <= set(df.columns)
        df = df.reindex(columns=list(COLUMNAS.values()))
        df["fila_excel"] = np.arange(2,len(df)+2)
        n_entrada = len(df)
        vacias = df[list(COLUMNAS.values())].isna().all(axis=1)
        tallos_entrada = pd.to_numeric(df.tallos,errors="coerce").sum()
        df = df.loc[~vacias].copy()
        df["fecha_corte"] = fechas(df.fecha_corte).dt.normalize()
        df["tallos"] = pd.to_numeric(df.tallos,errors="coerce").astype("Int64")
        df["finca"] = finca(df.finca)
        for col in ["producto","color","variedad"]:
            df[col] = normalizar(df[col])
        for col in ["invernadero","tipo_corte","grado","lote","estado_inventario","hora_corte","tipo_inventario"]:
            df[col] = df[col].astype("string").str.strip()
        for col in ["anio_fuente","semana_fuente"]:
            df[col] = pd.to_numeric(df[col],errors="coerce").astype("Int64")
        iso = df.fecha_corte.dt.isocalendar()
        df["semana_inicio"] = semana_operativa(iso.year,iso.week)
        df["archivo_origen"] = str(archivo.relative_to(ROOT)).replace("\\","/")
        df["hoja_origen"] = hoja
        df["llave_completa"] = df[["finca","producto","color","variedad","semana_inicio"]].notna().all(axis=1)
        assert df.tallos.sum() == tallos_entrada, "Se alteró la cantidad original"
        controles.append({"archivo":archivo.name,"hoja":hoja,"filas_entrada":n_entrada,
                          "vacias":int(vacias.sum()),"filas_limpias":len(df),"tallos":int(df.tallos.sum()),
                          "sin_fecha":int(df.fecha_corte.isna().sum()),"sin_tallos":int(df.tallos.isna().sum()),
                          "sin_llave":int((~df.llave_completa).sum())})
        partes.append(df)
        print(archivo.name,hoja,f"{len(df):,} filas / {int(df.tallos.sum()):,} tallos",flush=True)
    libro.close()
produccion = pd.concat(partes,ignore_index=True)
del partes, df
display(pd.DataFrame(controles))


Producción Acumulada 2021.xlsx Sheet1 926,798 filas / 70,932,003 tallos


Producción Acumulada 2022.xlsx Sheet1 965,024 filas / 66,543,676 tallos


Producción Acumulada 2023.xlsx Sheet1 1,042,035 filas / 72,438,033 tallos


Producción Acumulada 2023.xlsx Hoja1 95,268 filas / 6,501,739 tallos


Producción Acumulada 2024.xlsx Sheet1 1,029,587 filas / 72,044,273 tallos


Producción Acumulada 2024.xlsx sem 45 en adelante 166,767 filas / 11,665,544 tallos


Producción Acumulada 2025.xlsx 01-44 1,045,462 filas / 73,947,325 tallos


Producción Acumulada 2025.xlsx 45-52 178,455 filas / 12,566,939 tallos


Producción Acumulada 2026.xlsx BD 939,805 filas / 66,493,635 tallos


,archivo,hoja,filas_entrada,vacias,filas_limpias,tallos,sin_fecha,sin_tallos,sin_llave
0,Producción Acumulada 2021.xlsx,Sheet1,926798,0,926798,70932003,0,0,0
1,Producción Acumulada 2022.xlsx,Sheet1,965024,0,965024,66543676,0,0,0
2,Producción Acumulada 2023.xlsx,Sheet1,1042035,0,1042035,72438033,0,0,0
3,Producción Acumulada 2023.xlsx,Hoja1,95268,0,95268,6501739,0,0,0
4,Producción Acumulada 2024.xlsx,Sheet1,1029587,0,1029587,72044273,0,0,0
5,Producción Acumulada 2024.xlsx,sem 45 en adelante,930568,763801,166767,11665544,0,0,0
6,Producción Acumulada 2025.xlsx,01-44,1045462,0,1045462,73947325,0,0,0
7,Producción Acumulada 2025.xlsx,45-52,178455,0,178455,12566939,0,0,0
8,Producción Acumulada 2026.xlsx,BD,939835,30,939805,66493635,0,0,0


## 4. Imputación original, tipos y validaciones

Se reproduce la regla del benchmark recuperado: Tipo Corte faltante se completa con la moda de producto/color/variedad en las semanas anterior, actual y siguiente, dentro del mismo libro. En empates se aplica el desempate estable por Tipo Corte de la versión multianual. No usa finca en la llave de moda, igual que el original; no se interpreta como identificación física ni se usa en modelos del nuevo panel.

Se conserva `tipo_corte_original` y la bandera `tipo_corte_imputado`. **Esta es una reconstrucción retrospectiva que puede mirar la semana siguiente; no es feature conocida al origen.** Recuperar la limpieza original no autoriza fuga temporal en pronósticos.

Lote queda entero nullable; Hora Corte, texto HH:MM:SS. Ambos conservan la columna original. No se cambia un solo tallo ni se eliminan movimientos. La llave archivo/hoja/fila_excel sigue siendo única.


In [4]:
# Regla original del benchmark: moda de Tipo Corte en semana anterior, actual y siguiente.
# Se ejecuta por libro, como la limpieza multianual original. No modifica tallos.
produccion["tipo_corte_original"] = produccion.tipo_corte.copy()
produccion["tipo_corte"] = produccion.tipo_corte.replace({"":pd.NA,"<NA>":pd.NA})
produccion["_semana_calendario"] = produccion.fecha_corte.dt.to_period("W-SUN").dt.start_time
llave_moda = ["archivo_origen","producto","color","variedad","_semana_calendario"]
conteo = produccion.loc[produccion.tipo_corte.notna()].groupby(llave_moda+["tipo_corte"]).size().rename("cantidad").reset_index()
candidatos = pd.concat([conteo.assign(_semana_calendario=conteo._semana_calendario+pd.Timedelta(weeks=d)) for d in [-1,0,1]],ignore_index=True)
moda = (candidatos.groupby(llave_moda+["tipo_corte"],as_index=False).cantidad.sum()
        .sort_values(llave_moda+["cantidad","tipo_corte"],ascending=[True]*len(llave_moda)+[False,True],kind="stable")
        .drop_duplicates(llave_moda).set_index(llave_moda).tipo_corte)
mask = produccion.tipo_corte.isna()
claves = pd.MultiIndex.from_frame(produccion.loc[mask,llave_moda])
produccion.loc[mask,"tipo_corte"] = moda.reindex(claves).to_numpy()
produccion["tipo_corte_imputado"] = mask & produccion.tipo_corte.notna()
produccion = produccion.drop(columns="_semana_calendario")
print("Tipo Corte recuperado con la regla original:",int(produccion.tipo_corte_imputado.sum()),"; pendientes:",int(produccion.tipo_corte.isna().sum()))
# Tipos de datos equivalentes a la limpieza original, sin borrar el valor fuente.
produccion["lote_original"] = produccion.lote
lote_num = pd.to_numeric(produccion.lote,errors="coerce")
produccion["lote"] = lote_num.where(lote_num.mod(1).eq(0)).astype("Int64")
produccion["hora_corte_original"] = produccion.hora_corte
produccion["hora_corte"] = pd.to_datetime(produccion.hora_corte,format="mixed",errors="coerce").dt.strftime("%H:%M:%S").astype("string")
assert produccion.tallos.sum()==sum(c["tallos"] for c in controles)
assert not produccion.duplicated(["archivo_origen","hoja_origen","fila_excel"]).any()
assert produccion.tallos.notna().all()
assert produccion.fecha_corte.notna().all()
display(produccion.groupby("finca",dropna=False).agg(registros=("tallos","size"),tallos=("tallos","sum"),
                    desde=("fecha_corte","min"),hasta=("fecha_corte","max")))
display(produccion.isna().sum().to_frame("faltantes"))
display(produccion.dtypes.to_frame("tipo"))
gaitana = produccion.loc[produccion.finca.eq("GAITANA")]
assert gaitana.llave_completa.all()
print("Gaitana:",len(gaitana),"registros;",int(gaitana.tallos.sum()),"tallos")
print("Referencia de entrega anterior: 327.991.200 tallos; comparar, no recortar datos nuevos para igualarla.")


Tipo Corte recuperado con la regla original: 45906 ; pendientes: 2532204


,registros,tallos,desde,hasta
finca,,,,
<,1,70,2021-12-20,2021-12-20
ARABELLA,1734646,125141737,2021-01-04,2026-09-05
GAITANA,4654552,327991200,2021-01-04,2026-09-05
GAITANA PPTO,2,160,2021-04-28,2021-10-11


,faltantes
finca,0
invernadero,0
fecha_corte,0
anio_fuente,0
semana_fuente,0
tipo_corte,2532204
producto,0
grado,1598996
color,0
variedad,0


,tipo
finca,string
invernadero,string
fecha_corte,datetime64[us]
anio_fuente,Int64
semana_fuente,Int64
tipo_corte,string
producto,string
grado,string
color,string
variedad,string


Gaitana: 4654552 registros; 327991200 tallos
Referencia de entrega anterior: 327.991.200 tallos; comparar, no recortar datos nuevos para igualarla.


## 5. Exportación única y resumen

Reejecutar reemplaza este Parquet. No se generan CSV, archivos por año, versiones ni copias. No se modifica ningún Excel.

In [5]:
produccion.to_parquet(ARCHIVO_SALIDA,index=False)
print("EXPORTADO:",ARCHIVO_SALIDA)
print("Forma:",produccion.shape)
display(produccion.head())


EXPORTADO: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto\produccion_real_limpio.parquet
Forma: (6389201, 24)


,finca,invernadero,fecha_corte,anio_fuente,semana_fuente,tipo_corte,producto,grado,color,variedad,tallos,lote,estado_inventario,hora_corte,tipo_inventario,fila_excel,semana_inicio,archivo_origen,hoja_origen,llave_completa,tipo_corte_original,tipo_corte_imputado,lote_original,hora_corte_original
0,ARABELLA,Bloque 01,2021-01-04,2021,1,Corte 2,CARNATION,<NA>,BURGUNDY,ZURIGO,30,9910,C,14:55:00,<NA>,2,2021-01-04,Datos/Producción real/Producción Acumulada 202...,Sheet1,True,Corte 2,False,9910.0,2:55PM
1,ARABELLA,Bloque 01,2021-01-04,2021,1,Corte 2,CARNATION,<NA>,PURPLE,EXTASIS,8,15007,A,14:56:00,<NA>,3,2021-01-04,Datos/Producción real/Producción Acumulada 202...,Sheet1,True,Corte 2,False,15007.0,2:56PM
2,ARABELLA,Bloque 01,2021-01-04,2021,1,Corte 2,CARNATION,<NA>,BICOLOR BURGUNDY,OLYMPIA,32,18156,C,14:56:00,<NA>,4,2021-01-04,Datos/Producción real/Producción Acumulada 202...,Sheet1,True,Corte 2,False,18156.0,2:56PM
3,ARABELLA,Bloque 01,2021-01-04,2021,1,Corte 2,CARNATION,<NA>,YELLOW,DILETTA,12,36397,C,14:56:00,<NA>,5,2021-01-04,Datos/Producción real/Producción Acumulada 202...,Sheet1,True,Corte 2,False,36397.0,2:56PM
4,ARABELLA,Bloque 01,2021-01-04,2021,1,Corte 2,CARNATION,<NA>,GOLD,CAROLINE GOLD,23,16984,C,14:58:00,<NA>,6,2021-01-04,Datos/Producción real/Producción Acumulada 202...,Sheet1,True,Corte 2,False,16984.0,2:58PM
